In [1]:
import numpy as np
import random
import os
from skimage.io import imread 
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_addons as tfa
from batch_generators.BatchGeneratorTripletSemiHardLoss import BatchGenerator
from tensorflow.keras import layers, models
from keras.applications import DenseNet121
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from scipy.spatial import distance
import pickle
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

C:\Users\SOFIA\anaconda3\envs\env_tf_addons\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
path_images = "./dcface_0.5m_oversample_xid/dcface_0.5m_oversample_xid/images/"

NUM_IMGS = 1000
INPUT = (112, 112, 3)

In [3]:
data_files_train = [[path_images + str(id), id] for id in range(NUM_IMGS)]
batch_gen_train = BatchGenerator(data_files_train, dim = INPUT)

data_files_val = [[path_images + str(id), id] for id in range(NUM_IMGS, NUM_IMGS + 200)]
batch_gen_val = BatchGenerator(data_files_val, dim = INPUT)


In [4]:
#batch_gen_val.__getitem__(0)

## Model

In [5]:
backbone = DenseNet121(include_top = False,
                         weights = "imagenet",
                         input_shape = INPUT
                        )

for layer in backbone.layers:
    layer.trainable = False
                                            
inputs = layers.Input(shape=INPUT)

x = backbone(inputs)
x = tf.keras.layers.GlobalAveragePooling2D() (x)
dense1 = tf.keras.layers.Dense(units = 512, activation = 'relu')(x)
dense2 = tf.keras.layers.Dense(units = 256, activation = 'relu')(dense1)
outputs = tf.keras.layers.Dense(units = 256, activation = None)(dense2)

model = models.Model(inputs = inputs, outputs = outputs)

In [6]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(0.001),
    loss=tfa.losses.TripletSemiHardLoss(margin=0.5,
                                        distance_metric="angular"))

In [7]:
filepath="./models/model_semihard.keras"
checkpoint = ModelCheckpoint(filepath, 
                             monitor = 'val_loss', 
                             verbose = 1,
                             save_best_only = True, 
                             mode = 'min')

In [8]:
# Train the network
history = model.fit(
    batch_gen_train,
    validation_data=batch_gen_val,
    epochs=1000,
    callbacks=checkpoint)

Epoch 1/1000
31/31 [==============================] - ETA: 0s - loss: 0.4655
Epoch 1: val_loss improved from inf to 0.45701, saving model to ./models\model_semihard.keras
31/31 [==============================] - 75s 2s/step - loss: 0.4655 - val_loss: 0.4570
Epoch 2/1000
31/31 [==============================] - ETA: 0s - loss: 0.4500
Epoch 2: val_loss improved from 0.45701 to 0.44200, saving model to ./models\model_semihard.keras
31/31 [==============================] - 66s 2s/step - loss: 0.4500 - val_loss: 0.4420
Epoch 3/1000
31/31 [==============================] - ETA: 0s - loss: 0.4433
Epoch 3: val_loss did not improve from 0.44200
31/31 [==============================] - 60s 2s/step - loss: 0.4433 - val_loss: 0.4542
Epoch 4/1000
31/31 [==============================] - ETA: 0s - loss: 0.4574
Epoch 4: val_loss did not improve from 0.44200
31/31 [==============================] - 59s 2s/step - loss: 0.4574 - val_loss: 0.4569
Epoch 5/1000
31/31 [==============================] - ETA:

KeyboardInterrupt: 

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

## Prediction

In [ ]:
filepath = "./models/model_semihard.keras"
model.load_weights(filepath)

In [ ]:
with open('test_data.pkl', 'rb') as f:
    (true_pairs, true_pairs_1_test, true_pairs_2_test, false_pairs, false_pairs_1_test, false_pairs_2_test) = pickle.load(f)

In [ ]:
pred_true_pairs_1 = model.predict(true_pairs_1_test)
pred_true_pairs_2 = model.predict(true_pairs_2_test)

In [ ]:
real_test = np.concatenate((np.ones(len(true_pairs)), np.zeros(len(false_pairs))))
true_preds = [distance.cosine(pred_true_pairs_1[i], pred_true_pairs_2[i]) for i in range(len(pred_true_pairs_2))]

In [ ]:
pred_false_pairs_1 = model.predict(false_pairs_1_test)
pred_false_pairs_2 = model.predict(false_pairs_2_test)

In [ ]:
false_pred = [distance.cosine(pred_false_pairs_1[i], pred_false_pairs_2[i]) for i in range(len(pred_false_pairs_2))]

In [ ]:
def pred_threshold(true, false, thr):
    true = np.asarray([0 if val > thr else 1 for val in true])
    false = np.asarray([0 if val > thr else 1 for val in false])

    pred_test = np.concatenate((true, false))
    return pred_test

def metrics(y_test, y_pred):

    print('Acuraccy: %.3f' % accuracy_score(y_test, y_pred))
    print('Precision: %.3f' % precision_score(y_test, y_pred))
    print('Recall: %.3f' % recall_score(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.show()

In [ ]:
pred_test = pred_threshold(true_preds, false_pred, thr = 0.025)
metrics(real_test, pred_test)

In [ ]:
pred_test = pred_threshold(true_preds, false_pred, thr = 0.07)
metrics(real_test, pred_test)

In [ ]:
pred_test = pred_threshold(true_preds, false_pred, thr = 0.2)
metrics(real_test, pred_test)

In [ ]:
pred_test = pred_threshold(true_preds, false_pred, thr = 0.4)
metrics(real_test, pred_test)

In [ ]:
pred_test = pred_threshold(true_preds, false_pred, thr = 0.8)
metrics(real_test, pred_test)